pour 1 jour rmse= 2,24

In [ ]:
import pandas as pd
import numpy as np
import glob
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

# ==========================================
# 1. CHARGEMENT ET NETTOYAGE MASSIF
# ==========================================
files = sorted(glob.glob("synop_*.csv"))
cols_physiques = ['validity_time', 't', 'u', 'pres', 'dd', 'ff', 'td', 'pmer', 'tend', 'tend24', 'n', 'rr1', 'name']

print("⏳ Chargement des fichiers...")
df_raw = pd.concat([pd.read_csv(f, sep=None, engine='python', usecols=cols_physiques) for f in files])

df_raw['ds'] = pd.to_datetime(df_raw['validity_time'], utc=True)
df = df_raw[df_raw['name'] == "MONTPELLIER-AEROPORT"].drop_duplicates('ds').sort_values('ds').copy()

# Conversions et calculs de base
df['y'] = df['t'] - 273.15
df['dew_point'] = df['td'] - 273.15
df['pres'] = pd.to_numeric(df['pres'], errors='coerce') / 100

# Nettoyage des colonnes numériques
cols_num = ['y', 'u', 'pres', 'dd', 'ff', 'dew_point', 'pmer', 'tend', 'tend24', 'n', 'rr1']
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Interpolation (Correction Pandas 2.0+)
df = df.set_index('ds')[cols_num]
df['rr1'] = df['rr1'].fillna(0) # La pluie manquante = 0mm
df = df.interpolate(method='linear').bfill() 
df = df.resample('3h').interpolate(method='linear')

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
df['hour'] = df.index.hour
df['day_of_year'] = df.index.dayofyear

# Normales saisonnières (Base de 10 ans)
normales_map = df.groupby(['day_of_year', 'hour'])['y'].mean()
df['y_normal'] = [normales_map.get((d, h)) for d, h in zip(df['day_of_year'], df['hour'])]
df['residue'] = df['y'] - df['y_normal']

# Vecteurs de vent (U, V)
df['wind_u'] = df['ff'] * np.cos(np.deg2rad(df['dd']))
df['wind_v'] = df['ff'] * np.sin(np.deg2rad(df['dd']))

# Heure cyclique
df['hr_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hr_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

features = ['hr_sin', 'hr_cos', 'residue', 'u', 'pres', 'wind_u', 'wind_v', 'dew_point', 'pmer', 'tend', 'n', 'rr1']
df = df.dropna(subset=features)

# ==========================================
# 3. SCALING ET SEQUENCES
# ==========================================
scaler_X, scaler_y = MinMaxScaler(), MinMaxScaler()
scaled_X = scaler_X.fit_transform(df[features])
scaled_y = scaler_y.fit_transform(df[['residue']])

# Sauvegarde des scalers pour plus tard
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')

def create_sequences(data_X, data_y, in_len=72, out_len=8):
    X, y = [], []
    for i in range(len(data_X) - in_len - out_len):
        X.append(data_X[i:i+in_len])
        y.append(data_y[i+in_len : i+in_len+out_len, 0])
    return np.array(X), np.array(y)

X_data, y_data = create_sequences(scaled_X, scaled_y)
split = int(0.85 * len(X_data))
X_train, X_test = X_data[:split], X_data[split:]
y_train, y_test = y_data[:split], y_data[split:]

# ==========================================
# 4. MODÈLE LSTM
# ==========================================
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(100, return_sequences=True),
    Dropout(0.2),
    LSTM(50),
    Dense(32, activation='relu'),
    Dense(8)
])

model.compile(optimizer='adam', loss='mse')
print("🚀 Entraînement en cours (12 variables)...")
model.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1)

# ==========================================
# 5. PRÉDICTION (1er JANVIER 2026)
# ==========================================
cutoff = pd.Timestamp("2025-12-31 21:00:00", tz='UTC')
df_past = df[df.index <= cutoff].tail(72)
last_window = scaler_X.transform(df_past[features]).reshape(1, 72, len(features))

pred_res_scaled = model.predict(last_window)
pred_residue = scaler_y.inverse_transform(pred_res_scaled).flatten()

future_dates = pd.date_range(cutoff + pd.Timedelta(hours=3), periods=8, freq='3h')
future_normales = [normales_map.get((d.dayofyear, d.hour)) for d in future_dates]
final_forecast = pred_residue + np.array(future_normales)

# Export
df_res = pd.DataFrame({'ds': future_dates, 'y_pred': final_forecast})
df_res.to_csv("forecast_lstm_24h.csv", index=False)
model.save("modele_montpellier_final.keras")
print("\n✅ Prédictions sauvegardées dans 'forecast_lstm_24h.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Chargement
df_pred = pd.read_csv("forecast_lstm_24h.csv")
df_real = pd.read_csv("synop_2026.csv", sep=";")

df_pred['ds'] = pd.to_datetime(df_pred['ds'], utc=True)
df_real['ds'] = pd.to_datetime(df_real['validity_time'], utc=True)
df_real['y_real'] = df_real['t'] - 273.15
df_real = df_real[df_real['name'] == "MONTPELLIER-AEROPORT"].copy()

# 2. Alignement
df_compare = pd.merge(df_pred, df_real[['ds', 'y_real']], on='ds', how='inner').sort_values('ds')

if not df_compare.empty:
    mae = mean_absolute_error(df_compare['y_real'], df_compare['y_pred'])
    rmse = np.sqrt(mean_squared_error(df_compare['y_real'], df_compare['y_pred']))

    print(f"\n📊 RÉSULTATS DU 1er JANVIER 2026")
    print(f"MAE  : {mae:.2f} °C")
    print(f"RMSE : {rmse:.2f} °C")

    # 3. Graphique
    plt.figure(figsize=(12, 6))
    plt.plot(df_compare['ds'], df_compare['y_real'], 'o-', label='Réel', lw=2)
    plt.plot(df_compare['ds'], df_compare['y_pred'], 's--', label='LSTM Multi-varié', color='orange')
    plt.title(f"Comparaison Finale : MAE={mae:.2f} | RMSE={rmse:.2f}")
    plt.ylabel("Température (°C)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("❌ Erreur : Pas de données communes pour le 1er Janvier.")

Pour 1 semaine: rmse: 1,9

In [ ]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

# 1. PRÉPARATION PRO (On utilise StandardScaler pour mieux gérer les extrêmes)
files = sorted(glob.glob("synop_*.csv"))
df_raw = pd.concat([pd.read_csv(f, sep=None, engine='python', usecols=['validity_time', 't', 'u', 'pres', 'name']) for f in files])
df_raw['ds'] = pd.to_datetime(df_raw['validity_time'], utc=True)
df = df_raw[df_raw['name'] == "MONTPELLIER-AEROPORT"].drop_duplicates('ds').sort_values('ds').copy()

df['y'] = df['t'] - 273.15
df['u'] = pd.to_numeric(df['u'], errors='coerce')
df['pres'] = pd.to_numeric(df['pres'], errors='coerce') / 100

# Features Circulaires (Heure et Saison)
df['hour_sin'] = np.sin(2 * np.pi * df.ds.dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.ds.dt.hour / 24)
df['day_sin'] = np.sin(2 * np.pi * df.ds.dt.dayofyear / 365.25)
df['day_cos'] = np.cos(2 * np.pi * df.ds.dt.dayofyear / 365.25)

df = df.set_index('ds')[['y', 'u', 'pres', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']].interpolate().bfill()
df = df.resample('3h').interpolate()

# 2. ENTRAÎNEMENT
features = ['y', 'u', 'pres', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

X, y = [], []
in_len = 56 # On regarde 1 semaine en arrière
for i in range(len(scaled_data) - in_len):
    X.append(scaled_data[i:i+in_len])
    y.append(scaled_data[i+in_len, 0])

X, y = np.array(X), np.array(y)

model = Sequential([
    Input(shape=(in_len, len(features))),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mae') # MAE est plus robuste aux outliers que MSE
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.1, 
          callbacks=[EarlyStopping(patience=3, restore_best_weights=True)])

# 3. PRÉDICTION RÉCURSIVE
cutoff = pd.Timestamp("2025-12-31 21:00:00", tz='UTC')
current_window = scaled_data[df.index <= cutoff][-in_len:]
final_forecast = []

print("🔄 Calcul de la trajectoire thermique...")
for i in range(56):
    pred_scaled = model.predict(current_window.reshape(1, in_len, len(features)), verbose=0)[0, 0]
    
    # On dé-scale pour obtenir la température en °C
    temp_c = (pred_scaled * scaler.scale_[0]) + scaler.mean_[0]
    final_forecast.append(temp_c)
    
    # On prépare le point suivant (on simule l'heure et le jour pour les features circ)
    future_date = cutoff + pd.Timedelta(hours=3*(i+1))
    new_features = [
        pred_scaled, # La température prédite
        current_window[-1, 1], # On garde l'humidité
        current_window[-1, 2], # On garde la pression
        np.sin(2 * np.pi * future_date.hour / 24),
        np.cos(2 * np.pi * future_date.hour / 24),
        np.sin(2 * np.pi * future_date.dayofyear / 365.25),
        np.cos(2 * np.pi * future_date.dayofyear / 365.25)
    ]
    current_window = np.vstack([current_window[1:], new_features])

# Export
pd.DataFrame({'ds': pd.date_range(cutoff + pd.Timedelta(hours=3), periods=56, freq='3h'), 
              'y_pred': final_forecast}).to_csv("forecast_final.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Chargement
df_pred = pd.read_csv("forecast_final.csv")
df_real = pd.read_csv("synop_2026.csv", sep=";")

df_pred['ds'] = pd.to_datetime(df_pred['ds'], utc=True)
df_real['ds'] = pd.to_datetime(df_real['validity_time'], utc=True)
df_real['y_real'] = df_real['t'] - 273.15
df_real = df_real[df_real['name'] == "MONTPELLIER-AEROPORT"].copy()

df_compare = pd.merge(df_pred, df_real[['ds', 'y_real']], on='ds', how='inner').sort_values('ds')

if not df_compare.empty:
    mae = mean_absolute_error(df_compare['y_real'], df_compare['y_pred'])
    rmse = np.sqrt(mean_squared_error(df_compare['y_real'], df_compare['y_pred']))

    plt.figure(figsize=(15, 7))
    plt.plot(df_compare['ds'], df_compare['y_real'], 'o-', label='Réel (Janvier 2026)', color='#1f77b4', lw=2)
    plt.plot(df_compare['ds'], df_compare['y_pred'], 's--', label='LSTM Hybride (LSTM + Normales)', color='#ff7f0e')
    
    plt.title(f"Prévision 7 jours (Montpellier) | MAE : {mae:.2f}°C | RMSE : {rmse:.2f}°C")
    plt.ylabel("Température (°C)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()